In [0]:
data = """order_id,customer_id,customer_name,delivery_date,delivery_issue,delay_days,is_delayed
501,1001,Alice Smith,2026-06-14,Unreported Issue,12,1
502,1002,Bob Jones,2026-06-25,Sorting Hub Delay,1,1
503,1003,Charlie Brown,2026-06-16,Unreported Issue,10,1
504,1004,Diana Prince,2026-06-26,Damaged in Transit,0,0
506,1001,Alice Smith,2026-06-20,Unreported Issue,6,1
507,1002,Bob Jones,2026-06-27,Sorting Hub Delay,-1,0
509,1004,Diana Prince,2026-06-10,Unreported Issue,16,1
510,1005,Evan Wright,2026-06-24,Unreported Issue,2,1
511,1001,Alice Smith,2026-06-15,Sorting Hub Delay,11,1
512,1002,Bob Jones,2026-06-05,Unreported Issue,21,1
513,1003,Charlie Brown,2026-06-26,Incorrect Address,0,0
514,1004,Diana Prince,2026-06-18,Unreported Issue,8,1
515,1005,Evan Wright,2026-06-25,Damaged in Transit,1,1
"""

dbutils.fs.put("/tmp/orders.csv", data, overwrite=True)

Wrote 790 bytes.


True

In [0]:
# Extract the data
from pyspark.sql.functions import *

df = spark.read.csv("/tmp/orders.csv", header=True, inferSchema=True)
display(df)

order_id,customer_id,customer_name,delivery_date,delivery_issue,delay_days,is_delayed
501,1001,Alice Smith,2026-06-14,Unreported Issue,12,1
502,1002,Bob Jones,2026-06-25,Sorting Hub Delay,1,1
503,1003,Charlie Brown,2026-06-16,Unreported Issue,10,1
504,1004,Diana Prince,2026-06-26,Damaged in Transit,0,0
506,1001,Alice Smith,2026-06-20,Unreported Issue,6,1
507,1002,Bob Jones,2026-06-27,Sorting Hub Delay,-1,0
509,1004,Diana Prince,2026-06-10,Unreported Issue,16,1
510,1005,Evan Wright,2026-06-24,Unreported Issue,2,1
511,1001,Alice Smith,2026-06-15,Sorting Hub Delay,11,1
512,1002,Bob Jones,2026-06-05,Unreported Issue,21,1


In [0]:
# Transform the data
df = df.withColumn("latest_delivery_status", when(col("delay_days") > 0, "Delayed").when(col("delay_days") < 0, "Early Delivery").otherwise("Delivered On Time"))
print("Transformed data:")
display(df)

Transformed data:


order_id,customer_id,customer_name,delivery_date,delivery_issue,delay_days,is_delayed,latest_delivery_status
501,1001,Alice Smith,2026-06-14,Unreported Issue,12,1,Delayed
502,1002,Bob Jones,2026-06-25,Sorting Hub Delay,1,1,Delayed
503,1003,Charlie Brown,2026-06-16,Unreported Issue,10,1,Delayed
504,1004,Diana Prince,2026-06-26,Damaged in Transit,0,0,Delivered On Time
506,1001,Alice Smith,2026-06-20,Unreported Issue,6,1,Delayed
507,1002,Bob Jones,2026-06-27,Sorting Hub Delay,-1,0,Early Delivery
509,1004,Diana Prince,2026-06-10,Unreported Issue,16,1,Delayed
510,1005,Evan Wright,2026-06-24,Unreported Issue,2,1,Delayed
511,1001,Alice Smith,2026-06-15,Sorting Hub Delay,11,1,Delayed
512,1002,Bob Jones,2026-06-05,Unreported Issue,21,1,Delayed


In [0]:
# Load the data into a Delta table
df.write.format("delta").mode("overwrite").saveAsTable("orders")

In [0]:
# SQL query to show top 5 delayed customers
df.createOrReplaceTempView("v_orders")
display(spark.sql("SELECT customer_id, customer_name, COUNT(order_id) AS total_orders_placed, SUM(is_delayed) AS total_delayed_deliveries, MAX(delay_days) AS maximum_days_delayed FROM v_orders GROUP BY customer_id, customer_name ORDER BY total_delayed_deliveries DESC LIMIT 5"))

customer_id,customer_name,total_orders_placed,total_delayed_deliveries,maximum_days_delayed
1001,Alice Smith,3,3,12
1005,Evan Wright,2,2,2
1002,Bob Jones,3,2,21
1004,Diana Prince,3,2,16
1003,Charlie Brown,2,1,10
